# LN, classifier, and subunit-model tutorial

This notebook follows the full analysis sequence used by `demo_classifier_model_comparison.py`:

1. Load and crop the spike-triggered average (STA).
2. Fit a linear-nonlinear (LN) baseline on running noise.
3. Evaluate the LN model on frozen noise.
4. Train the spike/no-spike classifier.
5. Inspect its hidden-node spatial weights and select candidate subunits using Moran's I.
6. Build a rectified subunit model from those filters.
7. Compare LN and subunit-model predictions on the same frozen response.

Run this notebook from the repository root with the `rgc_subunit_classifier` Conda kernel. A small smoke configuration verifies the wiring but is not scientifically interpretable.

In [ ]:
from pathlib import Path
import json
import numpy as np
from matplotlib import pyplot as plt
from scipy.io import loadmat

from src.analysis.model_comparison import prediction_metrics
from src.prediction.sta_model_predictions import gen_sta_model, get_sta_predictions
from src.prediction.subunit_model_predictions import gen_subunit_model, get_subunit_predictions
from src.simulation.generate_training_data import crop_sta, gen_sta
from src.simulation.run_pipeline import run_subunit_model
from src.visualization.plotting_fxns import plot_predictions, plot_subunits

## Configuration

The defaults below are the full analysis settings. To perform a quick smoke demonstration, set `MAX_TRIALS = 2`, `NODE_NUM = 3`, `EPOCHS = 1`, and `MORANS_THRESHOLD = -1`. The permissive threshold only forces the smoke run to reach the final model stage.

In [ ]:
CELL = 0
DATA_PATH = Path('data/cell_data_01_NC.mat')
STIMULUS_PATH = Path('data/stimulus_data')
STA_PATH = Path('results/sta')
OUTPUT_PATH = Path('results/demo_classifier_model_comparison')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

NODE_NUM = 60
EPOCHS = 100
BATCH_SIZE = 25
LEARNING_RATE = 0.5
L1_COEFFICIENT = 1e-4
L2_COEFFICIENT = 0
SIGMA = 3
MORANS_THRESHOLD = 0.25
RANDOM_STATE = 0
MAX_TRIALS = None  # None uses all 219 trials

WIDTH, HEIGHT, REFRESH_RATE = 200, 150, 30
FILTER_LENGTH = int(0.67 * REFRESH_RATE)

## 1. Load the STA and define the receptive-field crop

The full STA is the spike-count-weighted mean stimulus history. Spatial and temporal slices pass through its largest absolute value. A fitted Gaussian ellipse limits model inputs to the receptive-field region.

In [ ]:
spike_data = loadmat(DATA_PATH)['spk1']
spikecounts = spike_data[CELL]
num_frames, total_trials = spikecounts.shape
num_trials = total_trials if MAX_TRIALS is None else min(MAX_TRIALS, total_trials)
spikecounts = spikecounts[:, :num_trials]

names = {
    'spatial_sta_filename': f'Cell {CELL} Uncropped Spatial STA.h5',
    'temp_sta_filename': f'Cell {CELL} Uncropped Temp STA.h5',
    'sta_filename': f'Cell {CELL} Uncropped STA.h5',
}
sta, spatial_sta, temporal_sta = gen_sta(
    WIDTH, HEIGHT, spikecounts, CELL, FILTER_LENGTH, num_frames, num_trials,
    sta_path=STA_PATH, stim_path=STIMULUS_PATH, **names,
)
cropped_sta, crop, ellipse_mask = crop_sta(spatial_sta, SIGMA)
crop_shape = cropped_sta.shape
stac = sta.reshape(WIDTH, HEIGHT, FILTER_LENGTH)[crop].reshape(cropped_sta.size, FILTER_LENGTH)

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
axes[0].imshow(spatial_sta.T, origin='lower', cmap='bwr')
axes[0].set_title('Spatial STA')
axes[1].plot(np.arange(FILTER_LENGTH) / REFRESH_RATE, temporal_sta)
axes[1].set_title('Temporal STA')
axes[1].set_xlabel('Time before spike (s)')
axes[2].imshow(cropped_sta.T, origin='lower', cmap='bwr')
axes[2].set_title(f'Cropped STA {crop_shape}')
fig.tight_layout()

## 2. Fit and evaluate the LN model

The linear stage filters each stimulus history with the cropped spatiotemporal STA. A three-parameter softplus nonlinearity maps that scalar projection to firing rate. Performance is measured against the mean response to repeated frozen noise.

In [ ]:
ln_parameters = gen_sta_model(
    WIDTH, HEIGHT, spikecounts, crop, *crop_shape, FILTER_LENGTH,
    num_frames, num_trials, stac, STIMULUS_PATH,
)
ln_predictions, ln_correlation, actual_firing_rate, time = get_sta_predictions(
    CELL, ln_parameters, stac, crop, *crop_shape, DATA_PATH, STIMULUS_PATH,
    filter_length=FILTER_LENGTH,
)
ln_metrics = prediction_metrics(
    actual_firing_rate, ln_predictions, n_parameters=3 + stac.size,
    filter_length=FILTER_LENGTH,
)
print(ln_metrics)
plot_predictions(ln_predictions, actual_firing_rate, time, FILTER_LENGTH, label='LN')

## 3. Train the classifier and examine hidden weights

The classifier predicts spike/no-spike from temporally filtered stimulus ensembles. Its first-layer weights are expanded from the ellipse pixels into full cropped spatial maps. Moran's I retains spatially coherent maps as candidate subunits.

In [ ]:
classifier = run_subunit_model(
    CELL, DATA_PATH, STIMULUS_PATH, sta_path=STA_PATH, node_num=NODE_NUM,
    batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, num_epochs=EPOCHS,
    l1_coefficient=L1_COEFFICIENT, l2_coefficient=L2_COEFFICIENT,
    sigma=SIGMA, random_state=RANDOM_STATE, max_trials=MAX_TRIALS,
    morans_threshold=MORANS_THRESHOLD,
)
print(f'Test accuracy: {classifier.test_accuracy:.3f}')
print(f'Selected {len(classifier.subunits)} of {NODE_NUM} hidden nodes')
plot_subunits(classifier.node_weights, crop_shape, columns=4);

In [ ]:
if len(classifier.subunits) == 0:
    raise RuntimeError(
        'No hidden weights exceeded the Moran threshold. Inspect the weights, '
        'train/tune the model, or explicitly justify another threshold.'
    )
plot_subunits(classifier.subunits, crop_shape, columns=3);

## 4. Build the subunit model

Each selected spatial filter is normalized. Its response is rectified, the subunit responses are combined to approximate the spatial STA, and a softplus output nonlinearity is fitted on running noise.

In [ ]:
subunit_parameters, combination_weights, normalized_subunits = gen_subunit_model(
    classifier.subunits, WIDTH, HEIGHT, spikecounts, FILTER_LENGTH,
    num_frames, num_trials, spatial_sta, temporal_sta, crop, *crop_shape,
    STIMULUS_PATH,
)
subunit_predictions, subunit_correlation, _, _ = get_subunit_predictions(
    CELL, subunit_parameters, combination_weights, normalized_subunits,
    temporal_sta, crop, *crop_shape, DATA_PATH, STIMULUS_PATH,
    filter_length=FILTER_LENGTH,
)
subunit_metrics = prediction_metrics(
    actual_firing_rate, subunit_predictions,
    n_parameters=3 + len(combination_weights), filter_length=FILTER_LENGTH,
)
print(subunit_metrics)

## 5. Compare held-out frozen-noise performance

Correlation and MSE are the clearest direct comparisons because both models predict the same time bins. AIC/BIC depend on the declared parameter-counting convention, especially whether learned spatial filters count as fitted parameters.

In [ ]:
comparison = {
    'LN': {'correlation': ln_metrics.correlation, 'mse': ln_metrics.mse,
           'aic': ln_metrics.aic, 'bic': ln_metrics.bic},
    'Subunit': {'correlation': subunit_metrics.correlation,
                'mse': subunit_metrics.mse, 'aic': subunit_metrics.aic,
                'bic': subunit_metrics.bic},
}
print(json.dumps(comparison, indent=2))

fig, ax = plt.subplots(figsize=(11, 4))
plot_predictions(ln_predictions, actual_firing_rate, time, FILTER_LENGTH, label='LN', ax=ax)
ax.plot(time[FILTER_LENGTH:], subunit_predictions, label='Subunit model')
ax.legend()
fig.tight_layout()

## Interpretation checklist

- Repeat training across several seeds and assess layout stability rather than trusting one run.
- Tune node count and regularization using held-out behavior, not visual appeal alone.
- Inspect both all hidden weights and the Moran-selected subset.
- Report the crop sigma, Moran threshold, trial count, random seed, and optimization settings.
- Compare LN and subunit models on exactly the same frozen-noise bins.
- Treat smoke-run outputs only as software verification.